# 03 - Feature Engineering: Chip Companies Financials

This notebook creates analytical features from the validated processed dataset. It does not modify either the raw data or financials_clean.csv.

In [1]:
from pathlib import Path
import pandas as pd

## Load and validate the clean input

In [2]:
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "data" / "processed").is_dir() else cwd.parent
input_path = project_root / "data" / "processed" / "financials_clean.csv"
output_path = project_root / "data" / "processed" / "financials_features.csv"
if not input_path.is_file():
    raise FileNotFoundError(f"Clean dataset not found: {input_path}")
df_clean = pd.read_csv(input_path)
df_features = df_clean.copy()
original_columns = df_clean.columns.tolist()
financial_columns = ["revenue_usd_bn", "operating_margin_pct", "operating_income_usd_bn", "rd_spend_usd_bn", "capex_usd_bn"]
input_validation = {
    "shape_617_by_20": df_clean.shape == (617, 20),
    "unique_year_company": not df_clean.duplicated(["year", "company_name"]).any(),
    "year_range_2010_2026": (int(df_clean["year"].min()), int(df_clean["year"].max())) == (2010, 2026),
    "original_financial_values_complete": not df_clean[financial_columns].isna().any().any(),
    "no_infinite_values": not df_clean.select_dtypes(include="number").isin([float("inf"), float("-inf")]).any().any(),
    "eur_review_flags_17": int(df_clean["country_code_review_flag"].sum()) == 17,
    "operating_income_review_flags_18": int(df_clean["operating_income_review_flag"].sum()) == 18,
}
for check, passed in input_validation.items():
    assert passed, f"Input validation failed: {check}"
print(f"Input path: {input_path}")
display(pd.Series(input_validation, name="passed").to_frame())

Input path: C:\Users\ALBERT\Desktop\Year 1 sem2\DADS5001_Aj.Thitirat_SUN\Mid-Term Project\semiconductor company datasets\data\processed\financials_clean.csv


,passed
shape_617_by_20,True
unique_year_company,True
year_range_2010_2026,True
original_financial_values_complete,True
no_infinite_values,True
eur_review_flags_17,True
operating_income_review_flags_18,True


## Value-chain grouping

In [3]:
def map_value_chain_group(segment):
    if segment == "foundry":
        return "Foundry"
    if segment.startswith("fabless_"):
        return "Fabless"
    if segment.startswith("idm_"):
        return "IDM"
    if segment.startswith("equipment_"):
        return "Equipment"
    if segment == "eda_software":
        return "EDA Software"
    return pd.NA

df_features["value_chain_group"] = df_features["segment"].map(map_value_chain_group).astype("string")
unmatched_segments = sorted(df_features.loc[df_features["value_chain_group"].isna(), "segment"].dropna().unique().tolist())
if unmatched_segments:
    raise ValueError(f"Unmatched segment values: {unmatched_segments}")
display(df_features[["segment", "value_chain_group"]].drop_duplicates().sort_values(["value_chain_group", "segment"]))

,segment,value_chain_group
493,eda_software,EDA Software
476,equipment_cleaning,Equipment
408,equipment_diversified,Equipment
425,equipment_etch,Equipment
442,equipment_inspection,Equipment
391,equipment_litho,Equipment
580,fabless_ai,Fabless
102,fabless_cpu_gpu,Fabless
136,fabless_diversified,Fabless
85,fabless_gpu,Fabless


## Growth and maturity features

Growth is available only for consecutive observations with a positive previous denominator. Undefined growth remains missing. first_observed_year is the first year visible for a company in this dataset, not necessarily its founding year.

In [4]:
df_features = df_features.sort_values(["company_name", "year"]).reset_index(drop=True)
company_group = df_features.groupby("company_name", sort=False)
df_features["previous_year"] = company_group["year"].shift(1).astype("Int64")
df_features["previous_revenue_usd_bn"] = company_group["revenue_usd_bn"].shift(1)
df_features["previous_operating_income_usd_bn"] = company_group["operating_income_usd_bn"].shift(1)
df_features["consecutive_year_flag"] = df_features["year"].sub(df_features["previous_year"]).eq(1).fillna(False)
valid_revenue_growth = df_features["consecutive_year_flag"] & df_features["previous_revenue_usd_bn"].gt(0)
df_features["revenue_growth_pct"] = pd.Series(float("nan"), index=df_features.index)
df_features.loc[valid_revenue_growth, "revenue_growth_pct"] = (df_features.loc[valid_revenue_growth, "revenue_usd_bn"].div(df_features.loc[valid_revenue_growth, "previous_revenue_usd_bn"]).sub(1).mul(100))
df_features["absolute_revenue_change_usd_bn"] = pd.Series(float("nan"), index=df_features.index)
df_features.loc[df_features["consecutive_year_flag"], "absolute_revenue_change_usd_bn"] = df_features.loc[df_features["consecutive_year_flag"], "revenue_usd_bn"] - df_features.loc[df_features["consecutive_year_flag"], "previous_revenue_usd_bn"]
valid_operating_growth = df_features["consecutive_year_flag"] & df_features["previous_operating_income_usd_bn"].gt(0)
df_features["operating_income_growth_pct"] = pd.Series(float("nan"), index=df_features.index)
df_features.loc[valid_operating_growth, "operating_income_growth_pct"] = (df_features.loc[valid_operating_growth, "operating_income_usd_bn"].div(df_features.loc[valid_operating_growth, "previous_operating_income_usd_bn"]).sub(1).mul(100))
df_features["first_observed_year"] = company_group["year"].transform("min").astype("int64")
df_features["years_since_first_observation"] = df_features["year"] - df_features["first_observed_year"]
df_features["growth_calculation_available"] = df_features["revenue_growth_pct"].notna()
display(df_features[["company_name", "year", "previous_year", "consecutive_year_flag", "revenue_growth_pct", "operating_income_growth_pct", "first_observed_year", "years_since_first_observation"]].head(10))

,company_name,year,previous_year,consecutive_year_flag,revenue_growth_pct,operating_income_growth_pct,first_observed_year,years_since_first_observation
0,AMD,2010,<NA>,False,NaN,NaN,2010,0
1,AMD,2011,2010,True,-4.255319,-20.661157,2010,1
2,AMD,2012,2011,True,-8.376068,6.250000,2010,2
3,AMD,2013,2012,True,-4.104478,-19.607843,2010,3
4,AMD,2014,2013,True,-12.451362,2.439024,2010,4
5,AMD,2015,2014,True,-12.000000,8.333333,2010,5
6,AMD,2016,2015,True,23.989899,29.670330,2010,6
7,AMD,2017,2016,True,14.867617,-14.406780,2010,7
8,AMD,2018,2017,True,14.539007,32.673267,2010,8
9,AMD,2019,2018,True,26.780186,2.238806,2010,9


## Shares, ranks, and investment features

Revenue shares describe only the companies represented in this dataset; they are not global market shares.

In [5]:
yearly_revenue = df_features.groupby("year")["revenue_usd_bn"].transform("sum")
df_features["revenue_share_within_dataset_pct"] = df_features["revenue_usd_bn"].div(yearly_revenue).mul(100)
df_features["revenue_rank_within_year"] = df_features.groupby("year")["revenue_usd_bn"].rank(method="dense", ascending=False).astype("int64")
df_features["value_chain_revenue_usd_bn"] = df_features.groupby(["year", "value_chain_group"])["revenue_usd_bn"].transform("sum")
df_features["value_chain_revenue_share_pct"] = df_features["value_chain_revenue_usd_bn"].div(yearly_revenue).mul(100)
positive_operating_income = df_features["operating_income_usd_bn"].where(df_features["operating_income_usd_bn"].gt(0))
df_features["rd_to_operating_income_pct"] = df_features["rd_spend_usd_bn"].div(positive_operating_income).mul(100)
df_features["capex_to_operating_income_pct"] = df_features["capex_usd_bn"].div(positive_operating_income).mul(100)
df_features["investment_ratio_available"] = df_features[["rd_to_operating_income_pct", "capex_to_operating_income_pct"]].notna().all(axis=1)
display(df_features[["year", "company_name", "value_chain_group", "revenue_share_within_dataset_pct", "revenue_rank_within_year", "value_chain_revenue_usd_bn", "value_chain_revenue_share_pct", "rd_to_operating_income_pct", "capex_to_operating_income_pct"]].head(10))

,year,company_name,value_chain_group,revenue_share_within_dataset_pct,revenue_rank_within_year,value_chain_revenue_usd_bn,value_chain_revenue_share_pct,rd_to_operating_income_pct,capex_to_operating_income_pct
0,2010,AMD,Fabless,2.600000,13,31.06,13.217021,100.826446,50.413223
1,2011,AMD,Fabless,2.341873,13,36.25,14.511609,121.875000,61.458333
2,2012,AMD,Fabless,2.009975,14,39.98,14.992313,104.901961,52.941176
3,2013,AMD,Fabless,1.808458,17,47.26,16.627964,125.609756,62.195122
4,2014,AMD,Fabless,1.537043,21,48.88,16.695700,107.142857,53.571429
5,2015,AMD,Fabless,1.271186,22,56.49,18.133667,86.813187,43.956044
6,2016,AMD,Fabless,1.329650,22,61.02,16.524494,83.050847,41.525424
7,2017,AMD,Fabless,1.310561,22,71.42,16.595794,111.881188,55.445545
8,2018,AMD,Fabless,1.356573,21,77.11,16.192776,96.268657,48.507463
9,2019,AMD,Fabless,1.654378,20,88.03,17.782042,119.708029,59.854015


## Completed feature validation

In [6]:
revenue_share_totals = df_features.groupby("year")["revenue_share_within_dataset_pct"].sum()
group_share_totals = (df_features[["year", "value_chain_group", "value_chain_revenue_share_pct"]].drop_duplicates().groupby("year")["value_chain_revenue_share_pct"].sum())
first_observation_counts = df_features["previous_year"].isna().groupby(df_features["company_name"]).sum()
zero_previous_revenue_mask = df_features["consecutive_year_flag"] & df_features["previous_revenue_usd_bn"].eq(0)
sorted_clean = df_clean.sort_values(["company_name", "year"]).reset_index(drop=True)
completed_validation = {
    "row_count_617": len(df_features) == 617,
    "unique_year_company": not df_features.duplicated(["year", "company_name"]).any(),
    "all_original_columns_present": set(original_columns).issubset(df_features.columns),
    "all_segments_grouped": df_features["value_chain_group"].notna().all(),
    "no_infinite_values": not df_features.select_dtypes(include="number").isin([float("inf"), float("-inf")]).any().any(),
    "yearly_revenue_shares_approximately_100": revenue_share_totals.sub(100).abs().le(1e-9).all(),
    "yearly_value_chain_shares_approximately_100": group_share_totals.sub(100).abs().le(1e-9).all(),
    "one_missing_previous_year_per_company": first_observation_counts.eq(1).all(),
    "zero_previous_revenue_growth_missing": df_features.loc[zero_previous_revenue_mask, "revenue_growth_pct"].isna().all(),
    "yearly_rank_starts_at_1": df_features.groupby("year")["revenue_rank_within_year"].min().eq(1).all(),
}
pd.testing.assert_frame_equal(df_features[original_columns].reset_index(drop=True), sorted_clean[original_columns], check_dtype=False, obj="original columns")
completed_validation["original_financial_values_unchanged"] = True
for check, passed in completed_validation.items():
    assert passed, f"Completed feature validation failed: {check}"
display(pd.Series(completed_validation, name="passed").to_frame())
display(pd.DataFrame({"revenue_share_total_pct": revenue_share_totals, "value_chain_share_total_pct": group_share_totals}))

,passed
row_count_617,True
unique_year_company,True
all_original_columns_present,True
all_segments_grouped,True
no_infinite_values,True
yearly_revenue_shares_approximately_100,True
yearly_value_chain_shares_approximately_100,True
one_missing_previous_year_per_company,True
zero_previous_revenue_growth_missing,True
yearly_rank_starts_at_1,True


,revenue_share_total_pct,value_chain_share_total_pct
year,,
2010,100.0,100.0
2011,100.0,100.0
2012,100.0,100.0
2013,100.0,100.0
2014,100.0,100.0
2015,100.0,100.0
2016,100.0,100.0
2017,100.0,100.0
2018,100.0,100.0


## Feature Engineering Summary

In [7]:
feature_summary = pd.DataFrame([
    ["value_chain_group", "High-level group mapped from segment", "category", "Unmatched segments stop execution", "Compare value-chain roles"],
    ["previous_year", "Previous observed year for the same company", "year", "Missing for first observation", "Diagnose continuity"],
    ["previous_revenue_usd_bn", "Previous observed company revenue", "USD billions", "Missing for first observation", "Growth denominator"],
    ["consecutive_year_flag", "Current year is exactly previous year plus one", "boolean", "Never missing", "Gate valid growth"],
    ["revenue_growth_pct", "Year-over-year revenue growth", "percent", "Missing unless consecutive and previous revenue > 0", "Measure revenue momentum"],
    ["absolute_revenue_change_usd_bn", "Revenue minus previous revenue", "USD billions", "Missing unless consecutive", "Measure absolute growth"],
    ["previous_operating_income_usd_bn", "Previous observed operating income", "USD billions", "Missing for first observation", "Growth denominator"],
    ["operating_income_growth_pct", "Year-over-year operating-income growth", "percent", "Missing unless consecutive and previous income > 0", "Measure earnings momentum"],
    ["revenue_share_within_dataset_pct", "Company share of yearly dataset revenue", "percent", "Missing if yearly denominator unavailable", "Measure dataset concentration"],
    ["revenue_rank_within_year", "Dense descending yearly revenue rank", "rank", "Never missing for valid revenue", "Compare company scale"],
    ["value_chain_revenue_usd_bn", "Yearly revenue summed by value-chain group", "USD billions", "Missing if group unavailable", "Measure group scale"],
    ["value_chain_revenue_share_pct", "Group share of yearly dataset revenue", "percent", "Missing if denominator unavailable", "Compare value-chain capture"],
    ["rd_to_operating_income_pct", "R&D divided by operating income", "percent", "Missing unless operating income > 0", "Assess investment relative to profit"],
    ["capex_to_operating_income_pct", "CapEx divided by operating income", "percent", "Missing unless operating income > 0", "Assess capital intensity relative to profit"],
    ["first_observed_year", "First company year visible in this dataset", "year", "Never missing", "Anchor observed tenure"],
    ["years_since_first_observation", "Years since first visible company year", "years", "Never missing", "Measure observed maturity"],
    ["growth_calculation_available", "Revenue growth is calculable", "boolean", "Never missing", "Filter valid growth rows"],
    ["investment_ratio_available", "Both investment-to-income ratios are calculable", "boolean", "Never missing", "Filter valid investment ratios"],
], columns=["feature_name", "definition", "unit", "missing_value_rule", "analytical_purpose"])
display(feature_summary)

,feature_name,definition,unit,missing_value_rule,analytical_purpose
0,value_chain_group,High-level group mapped from segment,category,Unmatched segments stop execution,Compare value-chain roles
1,previous_year,Previous observed year for the same company,year,Missing for first observation,Diagnose continuity
2,previous_revenue_usd_bn,Previous observed company revenue,USD billions,Missing for first observation,Growth denominator
3,consecutive_year_flag,Current year is exactly previous year plus one,boolean,Never missing,Gate valid growth
4,revenue_growth_pct,Year-over-year revenue growth,percent,Missing unless consecutive and previous revenu...,Measure revenue momentum
5,absolute_revenue_change_usd_bn,Revenue minus previous revenue,USD billions,Missing unless consecutive,Measure absolute growth
6,previous_operating_income_usd_bn,Previous observed operating income,USD billions,Missing for first observation,Growth denominator
7,operating_income_growth_pct,Year-over-year operating-income growth,percent,Missing unless consecutive and previous income...,Measure earnings momentum
8,revenue_share_within_dataset_pct,Company share of yearly dataset revenue,percent,Missing if yearly denominator unavailable,Measure dataset concentration
9,revenue_rank_within_year,Dense descending yearly revenue rank,rank,Never missing for valid revenue,Compare company scale


## Arrange, save, and re-read

In [8]:
ordered_columns = [
    "year", "company_name", "ticker", "country_iso3", "geo_code_type", "country_code_review_flag", "segment", "value_chain_group", "analysis_period", "source_validation_caution", "first_observed_year", "years_since_first_observation",
    "revenue_usd_bn", "operating_margin_pct", "operating_income_usd_bn", "rd_spend_usd_bn", "capex_usd_bn",
    "calculated_operating_income_usd_bn", "operating_income_abs_diff_usd_bn", "operating_income_review_flag", "rd_intensity_pct", "capex_intensity_pct", "zero_revenue_flag",
    "previous_year", "previous_revenue_usd_bn", "previous_operating_income_usd_bn", "consecutive_year_flag", "revenue_growth_pct", "absolute_revenue_change_usd_bn", "operating_income_growth_pct",
    "rd_to_operating_income_pct", "capex_to_operating_income_pct",
    "revenue_share_within_dataset_pct", "revenue_rank_within_year", "value_chain_revenue_usd_bn", "value_chain_revenue_share_pct",
    "growth_calculation_available", "investment_ratio_available",
]
df_features = df_features[ordered_columns]
df_features.to_csv(output_path, index=False, encoding="utf-8")
df_saved = pd.read_csv(output_path)
new_features = [column for column in ordered_columns if column not in original_columns]
saved_revenue_share_totals = df_saved.groupby("year")["revenue_share_within_dataset_pct"].sum()
saved_validation = {
    "shape": df_saved.shape == df_features.shape,
    "column_order": df_saved.columns.tolist() == ordered_columns,
    "duplicate_year_company_keys": int(df_saved.duplicated(["year", "company_name"]).sum()) == 0,
    "year_range": (int(df_saved["year"].min()), int(df_saved["year"].max())) == (2010, 2026),
    "value_chain_groups": set(df_saved["value_chain_group"].dropna()) == {"Foundry", "Fabless", "IDM", "Equipment", "EDA Software"},
    "no_infinite_values": not df_saved.select_dtypes(include="number").isin([float("inf"), float("-inf")]).any().any(),
    "revenue_share_totals": saved_revenue_share_totals.sub(100).abs().le(1e-9).all(),
}
for check, passed in saved_validation.items():
    assert passed, f"Saved-file validation failed: {check}"
print(f"Saved output: {output_path}")
print(f"Shape: {df_saved.shape}")
print(f"Year range: {df_saved['year'].min()}–{df_saved['year'].max()}")
print(f"Duplicate year-company keys: {df_saved.duplicated(['year', 'company_name']).sum()}")
print(f"Value-chain groups: {sorted(df_saved['value_chain_group'].unique().tolist())}")
display(df_saved[new_features].isna().sum().rename("missing_count").to_frame())
display(saved_revenue_share_totals.rename("revenue_share_total_pct").to_frame())
display(pd.Series(saved_validation, name="passed").to_frame())

Saved output: C:\Users\ALBERT\Desktop\Year 1 sem2\DADS5001_Aj.Thitirat_SUN\Mid-Term Project\semiconductor company datasets\data\processed\financials_features.csv
Shape: (617, 38)
Year range: 2010–2026
Duplicate year-company keys: 0
Value-chain groups: ['EDA Software', 'Equipment', 'Fabless', 'Foundry', 'IDM']


,missing_count
value_chain_group,0
first_observed_year,0
years_since_first_observation,0
previous_year,40
previous_revenue_usd_bn,40
previous_operating_income_usd_bn,40
consecutive_year_flag,0
revenue_growth_pct,48
absolute_revenue_change_usd_bn,40
operating_income_growth_pct,54


,revenue_share_total_pct
year,
2010,100.0
2011,100.0
2012,100.0
2013,100.0
2014,100.0
2015,100.0
2016,100.0
2017,100.0
2018,100.0


,passed
shape,True
column_order,True
duplicate_year_company_keys,True
year_range,True
value_chain_groups,True
no_infinite_values,True
revenue_share_totals,True


## Feature Engineering Conclusions

No rows were removed, and no original financial values were changed. Growth is calculated only for valid consecutive observations, and undefined growth is not replaced with zero. Revenue and value-chain shares refer only to companies represented in this dataset. The feature dataset is ready for EDA.